In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

2026-02-15 19:57:24.887389: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    "NWPU-RESISC45",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(224, 224),
    batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "NWPU-RESISC45",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224, 224),
    batch_size=32
)


Found 7000 files belonging to 10 classes.
Using 5600 files for training.


I0000 00:00:1771178260.650440  215285 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 749 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5


Found 7000 files belonging to 10 classes.
Using 1400 files for validation.


In [3]:
print(len(train_ds.class_names))
print(train_ds.class_names[:5])

10
['airplane', 'airport', 'baseball_diamond', 'basketball_court', 'beach']


In [5]:
model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(45, activation='softmax')
])


In [6]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [7]:
from tensorflow.keras.callbacks import ModelCheckpoint,EarlyStopping

checkpoint = ModelCheckpoint(
    filepath='best_img_model.keras',
    monitor='val_auc',
    verbose=1,
    save_best_only=True,
    mode='max'
)

earlystop = EarlyStopping(
    monitor='val_auc',
    patience=5,
    restore_best_weights=True,
    mode='max',
    verbose=1
)


In [8]:
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[checkpoint, earlystop]
)

Epoch 1/20


2026-02-15 17:33:59.572701: I external/local_xla/xla/service/service.cc:163] XLA service 0x725784005c10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-15 17:33:59.572754: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce GTX 1650, Compute Capability 7.5
2026-02-15 17:33:59.600815: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-15 17:33:59.883675: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91801
2026-02-15 17:34:02.618133: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[32,32,222,222]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,3,224,224]{3,2,1,0}, f32[32,3,3,3]{3,2,1,0}, f32[32]{0}), window={size=3x3}, dim_labels=bf01_oi01->bf01, custom_call_target="__c

  2/175 ━━━━━━━━━━━━━━━━━━━━ 12s 75ms/step - accuracy: 0.0234 - loss: 3.8683     

I0000 00:00:1771169648.825730  192385 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.2292 - loss: 2.4605

2026-02-15 17:34:22.101530: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[32,32,222,222]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,3,224,224]{3,2,1,0}, f32[32,3,3,3]{3,2,1,0}, f32[32]{0}), window={size=3x3}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kRelu","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]}
2026-02-15 17:34:22.232972: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[32,64,109,109]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,32,111,111]{3,2,1,0}, f32[64,32,3,3]{3,2,1,0}, f32[64]{0}), window={size=3x3}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActiv


Epoch 1: finished saving model to best_img_model.keras
175/175 ━━━━━━━━━━━━━━━━━━━━ 29s 104ms/step - accuracy: 0.3264 - loss: 1.9886 - val_accuracy: 0.5236 - val_loss: 1.2648
Epoch 2/20
  2/175 ━━━━━━━━━━━━━━━━━━━━ 13s 75ms/step - accuracy: 0.5312 - loss: 1.1782

/home/maxmaster/data/ITI/ai/.venv/lib/python3.12/site-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_auc` which is not available. Available metrics are: accuracy,loss,val_accuracy,val_loss
  current = self.get_monitor_value(logs)


175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.4983 - loss: 1.3766
Epoch 2: finished saving model to best_img_model.keras
175/175 ━━━━━━━━━━━━━━━━━━━━ 15s 88ms/step - accuracy: 0.5341 - loss: 1.3015 - val_accuracy: 0.6786 - val_loss: 0.9407
Epoch 3/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.6205 - loss: 1.0731
Epoch 3: finished saving model to best_img_model.keras
175/175 ━━━━━━━━━━━━━━━━━━━━ 15s 85ms/step - accuracy: 0.6352 - loss: 1.0393 - val_accuracy: 0.6236 - val_loss: 1.0018
Epoch 4/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.6883 - loss: 0.8775
Epoch 4: finished saving model to best_img_model.keras
175/175 ━━━━━━━━━━━━━━━━━━━━ 15s 86ms/step - accuracy: 0.7054 - loss: 0.8383 - val_accuracy: 0.6657 - val_loss: 0.9721
Epoch 5/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.7464 - loss: 0.7272
Epoch 5: finished saving model to best_img_model.keras
175/175 ━━━━━━━━━━━━━━━━━━━━ 15s 87ms/step - accuracy: 0.7634 - loss: 0.6926 - val_acc

In [4]:
from tensorflow.keras.optimizers import Adam


def build_model(num_filters1=32, num_filters2=64, num_filters3=128, dropout_rate=0.5, learning_rate=1e-3):
    input_shape = (224, 224, 3)
    num_classes = 10
    
    model = tf.keras.Sequential([
        layers.Rescaling(1./255, input_shape=input_shape),
        layers.Conv2D(num_filters1, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(num_filters2, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(num_filters3, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model


In [5]:

import random

search_space = {
    'num_filters1': [32, 64],
    'num_filters2': [64, 128],
    'num_filters3': [128, 256],
    'dropout_rate': [0.2, 0.3, 0.5],
    'learning_rate': [1e-3, 1e-4]
}

In [6]:


num_trials = 5

best_val_acc = 0
best_params = None

for trial in range(num_trials):
    # Randomly select hyperparameters
    params = {k: random.choice(v) for k,v in search_space.items()}
    print(f"\nTrial {trial+1}: {params}")
    
    # Build and train model
    model = build_model(**params)
    
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=5,
        verbose=1
    )
    
    val_acc = max(history.history['val_accuracy'])
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_params = params

print("\nBest hyperparameters:", best_params)
print("Best validation accuracy:", best_val_acc)

checkpointt = tf.keras.callbacks.ModelCheckpoint('best_model_resisc45.h5', save_best_only=True, monitor='val_loss')
earlystopp = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)



Trial 1: {'num_filters1': 32, 'num_filters2': 64, 'num_filters3': 256, 'dropout_rate': 0.3, 'learning_rate': 0.001}


/home/maxmaster/data/ITI/ai/.venv/lib/python3.12/site-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5


2026-02-15 19:57:55.758797: I external/local_xla/xla/service/service.cc:163] XLA service 0x738e240046b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-15 19:57:55.758836: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce GTX 1650, Compute Capability 7.5
2026-02-15 19:57:55.787184: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-15 19:57:56.084461: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91801
2026-02-15 19:57:57.191143: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[32,32,222,222]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,3,224,224]{3,2,1,0}, f32[32,3,3,3]{3,2,1,0}, f32[32]{0}), window={size=3x3}, dim_labels=bf01_oi01->bf01, custom_call_target="__c

UnknownError: Graph execution error:

Detected at node StatefulPartitionedCall defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/home/maxmaster/data/ITI/ai/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/maxmaster/data/ITI/ai/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/maxmaster/data/ITI/ai/.venv/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start

  File "/home/maxmaster/data/ITI/ai/.venv/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 211, in start

  File "/usr/lib/python3.12/asyncio/base_events.py", line 641, in run_forever

  File "/usr/lib/python3.12/asyncio/base_events.py", line 1987, in _run_once

  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run

  File "/home/maxmaster/data/ITI/ai/.venv/lib/python3.12/site-pac